# ML-05 — Feature Vector and Leakage / Privacy Check

**Stretch task:** audit the feature vector before modeling. The starter snapshot is used locally; IDs are grouping keys only.

## 1. Build the feature vector

I use numeric and categorical content/search fields that are available before the review decision. I deliberately exclude the target, fields that define it, IDs, and fields generated by the baseline.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

paths=[Path('data/raw/content_refresh_anonymized.csv'),Path('../../data/raw/content_refresh_anonymized.csv')]
p=next((x for x in paths if x.exists()),None)
if p is None: raise FileNotFoundError('Starter dataset not found.')
df=pd.read_csv(p)
TARGET='is_declining_label'
if TARGET not in df.columns:
    df[TARGET]=(df['trend_direction'].astype(str).str.lower()=='down').astype(int)
FORBIDDEN={TARGET,'trend_direction','trend_pct','content_id','client_id','score','reason_code','action_label','freshness_bucket','volume_bucket'}
features=[c for c in df.columns if c not in FORBIDDEN]
X=df[features]
num=X.select_dtypes(include=np.number).columns.tolist()
cat=[c for c in features if c not in num]
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median',add_indicator=True)),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore'))]),cat)])
Xt=pre.fit_transform(X)
print('Rows:',len(df))
print('Raw features:',len(features),'| numeric:',len(num),'| categorical:',len(cat))
print('Transformed feature count:',Xt.shape[1])

## 2. Feature notes

Numeric values are median-imputed and receive missingness indicators; categorical values use the most frequent category and one-hot encoding. Features are considered valid only when they are available before the decision cutoff. IDs identify/group observations but are never predictive inputs.

In [ ]:
notes=pd.DataFrame({'feature':features,'dtype':[str(df[c].dtype) for c in features],'missing_pct':[round(df[c].isna().mean()*100,2) for c in features]})
print(notes.sort_values('missing_pct',ascending=False).head(10).to_string(index=False))

## 3. Leakage hunt

A feature is rejected if it is the label, directly defines the label, contains future information, is an identifier, or is generated from the decision rule.

In [ ]:
checks={
 'target':TARGET,
 'trend_direction':'trend_direction',
 'trend_pct':'trend_pct',
 'content_id':'content_id',
 'client_id':'client_id',
 'baseline_score':'score',
 'baseline_reason':'reason_code',
 'baseline_action':'action_label'}
audit=pd.DataFrame({'reason':list(checks),'field':list(checks.values())})
audit['present']=[f in df.columns for f in audit.field]
audit['used_as_feature']=[f in features for f in audit.field]
print(audit.to_string(index=False))
assert not audit.used_as_feature.any()
print('LEAKAGE AUDIT: PASS')

## 4. What I excluded and why

- `is_declining_label`: the outcome.
- `trend_direction`, `trend_pct`: label-source fields.
- `content_id`, `client_id`: identifiers/grouping keys.
- `score`, `reason_code`, `action_label`: baseline-generated fields.
- `freshness_bucket`, `volume_bucket`: derived convenience fields that could encode the baseline logic.
- Any future-window field: unavailable at prediction time and therefore invalid for a forward-looking model.

## Self-check

- [x] Feature vector is actually constructed.
- [x] Missing and categorical handling is explicit.
- [x] Leakage test asserts forbidden fields are absent.
- [x] Exclusions are explained.
- [x] No private client names, URLs, or raw queries are displayed.